In [1]:
from parser.utils import parse_cfr_xml
from parser.types import Volume, Content
from typing import List

def extract_text_from_content(contents: List[Content]) -> str:
    texts = []
    for content in contents:
        if content.text:
            texts.append(content.text.strip())
        if content.subparagraphs:
            texts.extend([sub.text.strip() for sub in content.subparagraphs if sub.text])
    return "\n\n".join(texts)

def chunk_volume(volume: Volume):
    chunks = []
    for part in volume.parts:
        all_sections = part.sections.copy()
        for subpart in part.subparts:
            all_sections.extend(subpart.sections)

        for section in all_sections:
            if not section.content:
                continue

            chunk = {
                "volume": volume.metadata.title_number,
                "part": part.number,
                "section": section.number,
                "title": section.subject,
                "text": extract_text_from_content(section.content)
            }
            chunks.append(chunk)

    return chunks

In [2]:
parsed = parse_cfr_xml("title-12/CFR-2024-title12-vol1.xml")
volume = Volume.model_validate(parsed)
chunks = chunk_volume(volume)
try:
  vol1 = Volume.model_validate(parsed)
except Exception as e:
  print(e)
vol1.metadata

print(chunks[0])

{'volume': 'Title 12', 'part': '1', 'section': '§ 1.1', 'title': 'Authority, purpose, scope, and reservation of authority.', 'text': "Authority. This part is issued pursuant to 12 U.S.C. 1 et seq., 12 U.S.C. 24 (Seventh), and 12 U.S.C. 93a.\n\nPurpose This part prescribes standards under which national banks may purchase, sell, deal in, underwrite, and hold securities, consistent with the authority contained in 12 U.S.C. 24 (Seventh) and safe and sound banking practices.\n\nScope. The standards set forth in this part apply to national banks and Federal branches of foreign banks. Further, pursuant to 12 U.S.C. 335, State banks that are members of the Federal Reserve System are subject to the same limitations and conditions that apply to national banks in connection with purchasing, selling, dealing in, and underwriting securities and stock. In addition to activities authorized under this part, foreign branches of national banks are authorized to conduct international activities and inve

In [3]:
vol1.metadata

MetaData(title_number='Title 12', subject='Banks and Banking', parts='Parts 1 to 199', revised='Revised as of January 1, 2024', contains='Containing a codification of documents of general applicability and future effect', date='As of January 1, 2024', publication='Published by the Office of the Federal Register National Archives and Records Administration as a Special Edition of the Federal Register')